In [ ]:
# 'plural' expects a string, but if the user leaves it blank, it defaults to an empty string ''
def show_count(count: int, singular: str, plural: str = '') -> str:
    
    # 1. Handle the singular case first
    if count == 1:
        return f'1 {singular}'
    
    # 2. Format the number (turns 0 into 'no')
    count_str = str(count) if count else 'no'
    
    # 3. Check if the user provided a custom plural word
    if not plural:
        # If they left it blank, do the lazy thing and just add an 's'
        plural = singular + 's'
        
    return f'{count_str} {plural}'


# Works normally for singular
print(show_count(1, 'mouse'))          
# Output: '1 mouse'

# Works normally for regular plurals (leaves 3rd argument blank)
print(show_count(3, 'dog'))            
# Output: '3 dogs'

# Fixes the weird plurals (provides the 3rd argument)
print(show_count(3, 'mouse', 'mice'))  
# Output: '3 mice'

1 mouse
3 dogs
3 mice


if your parameter expects a list or a dictionary (mutable types), setting the default to an empty list [] is one of the most dangerous traps in Python. If you do that, every time you call the function, it will share the exact same list, causing massive bugs.

In [4]:
from typing import Optional
def show_count(count: int, singular: str, plural: Optional[str] = None) -> str:
    if count == 1:
        return f"1 {singular}"
    if plural is None:
        plural = singular + 's'
    return f"{count} {plural}"

# Works normally for singular
print(show_count(1, 'mouse'))          
# Output: '1 mouse'

# Works normally for regular plurals (leaves 3rd argument blank)
print(show_count(3, 'dog'))            
# Output: '3 dogs'

# Fixes the weird plurals (provides the 3rd argument)
print(show_count(3, 'mouse', 'mice'))  
# Output: '3 mice'

1 mouse
3 dogs
3 mice


difference between how Python actually runs and how Type Checkers think it runs.

The core idea here is: In Python, a "type" isn't defined by its name. It's defined by what it can DO.

In [ ]:
def double(x):
    return x * 2


What type is x?

Is it an integer (5 * 2 = 10)?

Is it a string ("ha" * 2 = "haha")?

Is it a list ([1] * 2 = [1, 1])?

In Python, the answer is: Who cares? As long as the object has the ability to multiply itself (specifically, if it has a __mul__ method under the hood), Python will happily run the code. Python doesn't care about the name of the type, it only cares if the operation (* 2) is supported

In [2]:
from collections import abc

# We add a type hint saying 'x' must be a Sequence (like a list, string, or tuple)
def double(x: abc.Sequence):
    return x * 2  # MYPY WILL FLAG THIS AS AN ERROR!

why does Mypy throw an error? We just proved that strings and lists (which are Sequences) can be multiplied by 2!

Here is why: Type checkers only read the blueprints, they don't run the code.
When you tell Mypy that x is an abc.Sequence (the abstract blueprint for all sequences), Mypy looks at that specific blueprint. The official blueprint for a Sequence promises that you can find its length and look at its items, but it does not officially promise that it can be multiplied.

Even though in real life (at runtime) a string can be multiplied, Mypy is strictly looking at the abc.Sequence label you slapped on it. Because the label doesn't explicitly guarantee multiplication, Mypy yells at you.

The Two Worldviews: Duck Typing vs. Nominal Typing
This mismatch brings us to the two completely different ways of viewing types in a "gradually typed" language like Python.

ــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــ

Duck Typing (How Python actually runs)
"If it walks like a duck and quacks like a duck, it's a duck."

How it works: Variables are just empty boxes. The objects you put inside them have types. Python doesn't look at the label on the box; it just reaches inside, grabs the object, and tries to make it .quack().

The Pros: It is incredibly flexible. You can pass almost anything into a function as long as it supports the operation you want to do.

The Cons: If you pass in a Cat by accident, Python won't realize the mistake until it actually tries to make the Cat .quack(), and then your program crashes while it's running.

ــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــ

Nominal Typing (How Type Checkers read your code)
"I only care about the label on the box."

How it works: This is how strict languages (like Java) and Mypy operate. You put a strict label on the variable (the box), like birdie: Bird.

The Catch: Let's say you create a specific Duck object, put it in the box, and call birdie.quack(). Mypy will throw an error! Why? Because the label on the box just says Bird, and generic birds don't quack. Mypy completely ignores the fact that there is a real Duck inside the box. It only trusts the Bird label.

The Pros: Mypy catches errors by just reading your code, before you ever hit "Run" (catching bugs early).

The Cons: It is rigid and annoying. You have to be incredibly precise with your labels, otherwise the type checker will reject code that would actually run perfectly fine.

In [5]:
# A ridiculously simple function. 
# Notice there are ZERO type hints. We don't say if a and b are numbers, words, or whatever.
def add_things_together(a, b):
    return a + b

# --- 1. Standard Math (Integers and Floats) ---
print(add_things_together(5, 10))        
# Output: 15
print(add_things_together(3.5, 2.1))     
# Output: 5.6

# --- 2. String Math (Concatenation) ---
print(add_things_together("Hello ", "World")) 
# Output: 'Hello World'

# --- 3. List Math (Merging) ---
print(add_things_together([1, 2], [3, 4]))    
# Output: [1, 2, 3, 4]


# --- 4. THE ULTIMATE DUCK TYPING TEST (Custom Objects) ---

class Wallet:
    def __init__(self, dollars):
        self.dollars = dollars
        
    # This magic method tells Python how to use the math '+' symbol on this object
    def __add__(self, other_wallet):
        return self.dollars + other_wallet.dollars

# Create two entirely custom objects that the function has never seen before
my_wallet = Wallet(50)
your_wallet = Wallet(100)

# Pass them into our generic math function
print(add_things_together(my_wallet, your_wallet))
# Output: 150

15
5.6
Hello World
[1, 2, 3, 4]
150


In [6]:
# We put a sticky note on 'a' and 'b' saying they are generic "objects"
def add_things(a: object, b: object):
    return a + b  # <--- MYPY THROWS A MASSIVE ERROR HERE

my_wallet = Wallet(50)
your_wallet = Wallet(100)

add_things(my_wallet, your_wallet)

150

1. Mypy's Brain (Nominal Typing)
Mypy does not care that you passed in a Wallet. It doesn't even look at the Wallet class.

Mypy looks at the sticky note: a: object.

Mypy pulls up the official blueprint for a standard Python object.

Mypy checks: "Does the official object blueprint have an __add__ method?"

The answer is No.

Mypy throws an error: Unsupported operand type for +: 'object' and 'object'.

2. Python's Brain (Duck Typing / Runtime)
Python completely ignores the sticky notes.

You run the code.

Python grabs the actual items out of the boxes (the two Wallet objects).

Python tries to push them together using +.

Python sees that the Wallet object has __add__ defined, runs the math, and happily spits out 150 without a single crash.

How do you fix the Mypy error?
To make Mypy happy, the "sticky note" has to explicitly guarantee that the + symbol exists. If you change the type hint to Wallet, the error instantly goes away:

In [7]:
# Now the sticky note perfectly matches the object
def add_things(a: Wallet, b: Wallet):
    return a + b  # <--- MYPY IS HAPPY!

In [9]:
# What you write:
def double(x):
    return x * 2

# How the Type Checker reads it:
# def double(x: Any) -> Any:
#     return x * 2

# ypu can't do :object because that means it has the basic stuff like if it exists but Any means it can has the add the mul and so

The Subtype Rule (Liskov Substitution Principle)
To understand how strict type checkers work, you need to understand the Liskov Substitution Principle (LSP). Barbara Liskov essentially defined the golden rule of Object-Oriented Programming:

"If it needs a generic thing, you can give it a specific thing. But if it needs a specific thing, you cannot give it a generic thing."

Let's use T1 (Vehicle) and T2 (Car). A Car is a subclass of Vehicle.

Allowed: If a function asks for a Vehicle (T1), you can pass it a Car (T2). Why? Because a Car inherited everything a Vehicle can do (drive, stop). It will work perfectly.

Illegal: If a function asks for a Car (T2), you cannot pass it a generic Vehicle (T1). Why? Because a generic Vehicle might be a bicycle. If the function tries to turn on the air conditioning (a Car-specific operation), the bicycle will crash the program.

In [10]:
# 1. We create a generic Parent class
class Vehicle:
    def drive(self):
        print("Moving forward")

# 2. We create a specific Child class that inherits from Vehicle
class Car(Vehicle):
    def turn_on_ac(self):
        print("AC is blowing cold air")

# --- THE TESTS ---

# Function A expects a generic Vehicle
def wash_vehicle(v: Vehicle):
    v.drive()

# Function B strictly expects a Car
def cool_down_car(c: Car):
    c.turn_on_ac()


my_generic_ride = Vehicle()
my_civic = Car()

# ✅ LEGAL: A Car is a Vehicle. It knows how to drive.
wash_vehicle(my_civic)

# ❌ ILLEGAL (Mypy Error): A generic Vehicle doesn't necessarily have AC! 
# If Python actually ran this, it would crash when it tried to call .turn_on_ac()
cool_down_car(my_generic_ride)

Moving forward


AttributeError: 'Vehicle' object has no attribute 'turn_on_ac'

In [11]:
from typing import Any

class Car:
    pass

my_civic = Car()
my_word = "Hello"
my_number = 42

# --- Rule 2: Everything fits into Any ---
# This function doesn't care what you give it.
def print_anything(item: Any):
    print(item)

# ✅ LEGAL: Mypy accepts all of these because the parameter is Any.
print_anything(my_civic)
print_anything(my_word)
print_anything(my_number)


# --- Rule 3: Any fits into everything ---

# This function STRICTLY wants a Car object.
def repair_car(c: Car):
    pass

# We create a mystery variable and explicitly tell Mypy it is 'Any'
mystery_box: Any = "I am literally a string, not a car"

# ✅ LEGAL (Wait, what?!): 
# Mypy will NOT throw an error here! Because mystery_box is labeled 'Any', 
# Mypy turns a blind eye and says, "Well, it's Any, so it MIGHT be a Car. I'll allow it."
# (Note: Python will still crash at runtime if repair_car tries to do Car things to a string).
repair_car(mystery_box)

Hello
42


In [12]:
# this is called infers where it right from top to down

# We don't write any type hints here.
# Mypy looks at len(), knows that counting letters always results in an integer, 
# and secretly assigns 'x' the type of 'int'.
x = len("hello world")

# ✅ LEGAL: Mypy knows 'x' is an integer, so math is allowed.
y = x * 10 

# ❌ ILLEGAL (Mypy Error): Mypy knows 'x' is an integer, and integers don't have uppercase letters!
# Error: "int" has no attribute "upper"
bad_idea = x.upper()

AttributeError: 'int' object has no attribute 'upper'

In [13]:
# This function strictly asks for a float (decimal)
def calculate_tax(price: float) -> float:
    return price * 1.20

# ✅ LEGAL: Even though 100 is an int, Mypy allows it because of the math exception.
# It automatically treats it as 100.0
calculate_tax(100)

120.0

In [14]:
from typing import Union

# The return type is a Union. It will spit out EITHER a float OR a string.
def parse_token(token: str) -> Union[str, float]:
    try:
        # If the string is "3.14", it converts it to a float and returns it
        return float(token)
    except ValueError:
        # If the string is "hello", math fails, so it just returns "hello"
        return token

In [15]:
# --- OLD WAY (Pre-3.10) ---
from typing import Union, Optional

def process_data(data: Union[str, bytes], default: Optional[str] = None):
    pass

# --- NEW WAY (Python 3.10+) ---
# No imports needed! The '|' does all the work.

def process_data(data: str | bytes, default: str | None = None):
    pass

In [16]:
# The -> list[str] tells Mypy: "This function returns a list, and EVERY item inside it is a string."
def tokenize(text: str) -> list[str]:
    # .split() takes "hello world" and returns ['HELLO', 'WORLD']
    return text.upper().split()

# If you just write 'list' without the brackets, Mypy assumes you mean 'list[Any]'.
# This means it goes back to being a free-for-all box.
def bad_function() -> list:
    return [1, "two", 3.0] # Mypy won't catch this mess

In [17]:
# Mypy knows this tuple MUST have exactly two floats, no more, no less.
def get_location() -> tuple[float, float]:
    return (31.2304, 121.4737)

# This tuple MUST be a string, then a float, then a string.
city_data: tuple[str, float, str] = ('Shanghai', 24.28, 'China')

In [18]:
# If you have a tuple with a lot of fields (like 5 or 6 items), remembering that index [3] is the Zip Code is annoying.

from typing import NamedTuple

# We create our blueprint
class Coordinate(NamedTuple):
    lat: float
    lon: float

# We just use 'Coordinate' as the type hint!
def geohash(lat_lon: Coordinate) -> str:
    pass

In [19]:
# The '...' tells Mypy: "This tuple can be 1 item long, or 10,000 items long, 
# but EVERY item inside it must be an integer."
winning_lottery_numbers: tuple[int, ...] = (4, 8, 15, 16, 23, 42)

In [ ]:
from collections.abc import Mapping
from collections import UserDict

# ❌ THE RIGID WAY (Bad)
def process_data(data: dict[str, int]):
    pass

# ✅ THE FLEXIBLE WAY (Good)
def process_data(data: Mapping[str, int]):

    pass


If you use dict as your type hint, Mypy will only accept a standard Python dictionary. If another developer builds a custom dictionary using UserDict (which is the recommended way to build custom dictionaries in Python), Mypy will reject it! Why? Because UserDict and dict are siblings; one does not inherit from the other.

In [21]:
from collections.abc import Iterable, Sequence

# 1. ITERABLE EXAMPLE
# We just loop through. We don't care how long it is. 
# This is safe to use with generators!
def print_all(items: Iterable[str]):
    for item in items:
        print(item)

# 2. SEQUENCE EXAMPLE
# We MUST use Sequence here because we are using len() and slicing [::]. 
# If we passed an Iterable (like a generator) into this, it would crash!
def split_in_half(items: Sequence[str]):
    midpoint = len(items) // 2
    return items[:midpoint], items[midpoint:]

In [22]:
from typing import TypeVar
from collections.abc import Sequence

# We create a placeholder type and call it 'T'
T = TypeVar('T')

# The signature now says: 
# "Whatever type 'T' goes into this function, a list of that exact same type 'T' comes out."
def sample(population: Sequence[T], size: int) -> list[T]:
    pass

In [23]:
from typing import TypeVar
from collections.abc import Hashable

# This placeholder accepts ANYTHING, as long as it is a subclass of Hashable.
HashableT = TypeVar('HashableT', bound=Hashable)

def mode(data: list[HashableT]) -> HashableT:
    pass

In [24]:
from typing import AnyStr

# AnyStr is literally just: TypeVar('AnyStr', str, bytes)

def process_text(text: AnyStr) -> AnyStr:
    return text

### his is one of the most exciting additions to Python's typing system. This section bridges the gap between Python's flexible nature (Duck Typing) and Mypy's strict rules (Nominal Typing). The concept here is called Static Duck Typing via typing.Protocol

In [25]:
from typing import Protocol

# 1. WE CREATE THE CHECKLIST (The Protocol)
# We just define the skill we want. The "..." means we don't write the actual code here.
class CanFly(Protocol):
    def fly(self) -> str: ...


# 2. WE MAKE TWO COMPLETELY UNRELATED CLASSES
class Bird:
    def fly(self) -> str:
        return "Flap flap!"

class Airplane:
    def fly(self) -> str:
        return "Jet engines go brrrrr!"


# 3. WE HIRE THE BOUNCER (The Function)
# We tell the function: "You will accept anything that matches the 'CanFly' checklist."
def launch_into_sky(flying_thing: CanFly):
    print(flying_thing.fly())


# --- LET'S TEST IT ---

my_bird = Bird()
my_plane = Airplane()

# ✅ Mypy allows this! (Because Bird has a fly method)
launch_into_sky(my_bird)  

# ✅ Mypy ALSO allows this! (Because Airplane has a fly method)
launch_into_sky(my_plane)

Flap flap!
Jet engines go brrrrr!


In [26]:
from collections.abc import Callable

# We define a function that does addition
def add_numbers(a: int, b: int) -> int:
    return a + b

# THE TYPE HINT:
# 'math_operation' MUST be a function. 
# It MUST take two integers. It MUST return one integer.
def calculator(x: int, y: int, math_operation: Callable[[int, int], int]) -> int:
    # We execute the function they passed in
    return math_operation(x, y)

# We pass the 'add_numbers' function directly into 'calculator'
print(calculator(5, 10, add_numbers)) 
# Output: 15

15


The Variance Mind-Bender (Outputs vs. Inputs)
This is where things get trippy. What happens if the function you pass in doesn't exactly match the Callable type hint?

Let's say our type hint asks for a function that deals with float (decimals).
Remember our math rule from earlier: int is a subtype of float, which is a subtype of complex.

Mypy treats the Outputs (returns) and the Inputs (arguments) of a Callable completely differently.

Rule 1: Outputs are Flexible Downwards (Covariance)
If Mypy expects a function to return a float, it is perfectly safe to pass a function that returns an int.

Why? Because if my system is expecting a decimal like 5.0, and your function hands it a solid 5, my system won't crash. It can easily treat an integer as a decimal.


Rule 2: Inputs are Flexible UPWARDS (Contravariance)
This is the trap! If Mypy expects a function that takes a float as an argument, you CANNOT pass it a function that only takes an int.

Why? Because update_system is going to hand your function a decimal (like 3.14). If your function strictly requires an integer, it will choke on the decimal and crash.

However, you can pass a function that takes a complex number. Why? Because a function designed to handle massive, complicated complex numbers can easily process a simple float without breaking a sweat.

In [27]:
# The hint '*toppings: str' means every single item you pass in MUST be a string.
def make_pizza(*toppings: str):
    
    # INSIDE the function, Mypy knows that 'toppings' is actually a tuple[str, ...]
    for item in toppings:
        print(f"Adding {item}")

# ✅ LEGAL: We pass in individual strings
make_pizza("pepperoni", "mushrooms", "onions") 

# ❌ ILLEGAL: Mypy catches the integer
make_pizza("pepperoni", 5)

Adding pepperoni
Adding mushrooms
Adding onions
Adding pepperoni
Adding 5


In [28]:
# The hint '**options: bool' means every value passed in MUST be a boolean.
def configure_app(**options: bool):
    
    # INSIDE the function, Mypy knows 'options' is actually a dict[str, bool]
    if options.get("dark_mode"):
        print("Dark mode on!")

# ✅ LEGAL: The values (True/False) match the boolean type hint
configure_app(dark_mode=True, notifications=False)

# ❌ ILLEGAL: "blue" is a string, not a boolean!
configure_app(dark_mode=True, theme_color="blue")

Dark mode on!
Dark mode on!


In [29]:
def greet(name: str, /, greeting: str = "Hello"):
    print(f"{greeting} {name}")

# ✅ LEGAL: We just pass the string "Alice" by its position.
greet("Alice", greeting="Hi")

# ❌ ILLEGAL (Python will crash!): You are NOT allowed to use the keyword 'name=' 
# because it came before the slash.
greet(name="Alice", greeting="Hi")

Hi Alice


TypeError: greet() got some positional-only arguments passed as keyword arguments: 'name'

Type Hints (Static Checking): Used to help developers write code faster, catch silly typos in the editor, and keep the program running at maximum speed because they disappear at runtime.

isinstance() / if statements (Runtime Checking): Used to protect your program from unpredictable garbage data coming from actual users or external APIs, at the cost of slightly slowing down your code.

In [32]:
# --- 1. UNPACKING INTO VARIABLES ---
my_grades = [95, 82, 75, 99, 88]

# We want the first grade, the last grade, and we don't care about the middle.
# The '*' dynamically packs all the leftover middle items into 'middle_grades'
first, *middle_grades, last = my_grades

print(first)         # Output: 95
print(middle_grades) # Output: [82, 75, 99]
print(last)          # Output: 88


# --- 2. UNPACKING INTO FUNCTIONS (Where Mypy struggles) ---
def create_profile(name, age, city):
    print(f"{name} is {age} and lives in {city}")

# We pull this data from a database or API
user_data = {"name": "Alice", "age": 30, "city": "New York"}

# Instead of writing: create_profile(user_data["name"], user_data["age"], user_data["city"])
# We just use '**' to explode the dictionary directly into the function arguments!
create_profile(**user_data)

95
[82, 75, 99]
88
Alice is 30 and lives in New York


## Metaclasses (The "Dark Magic" of Python)
The Definition:
You already know that a Class (like Car) is a blueprint for creating an Object (like my_honda).
Well, a Metaclass is a blueprint for creating a Class.

Metaclasses allow you to write code that automatically alters or injects features into a class the exact moment the class is written. It is mostly used by developers who build massive frameworks (like Django or SQLAlchemy) to hide complex setup code from the users

In [33]:
# 1. WE BUILD THE METACLASS (The blueprint of blueprints)
# We inherit from 'type', which is the default metaclass in Python.
class StrictBossMeta(type):
    
    # The __new__ method runs the split-second a new class is being created
    def __new__(cls, name, bases, attrs):
        uppercase_attrs = {}
        
        # We loop through everything the developer wrote in their class
        for key, value in attrs.items():
            if not key.startswith('__'):  # Ignore Python's built-in magic methods
                # Force the variable name to be UPPERCASE
                uppercase_attrs[key.upper()] = value
            else:
                uppercase_attrs[key] = value
                
        # We construct and return the modified class
        return super().__new__(cls, name, bases, uppercase_attrs)


# 2. THE DEVELOPER WRITES THEIR CLASS (and uses our Metaclass)
class Employee(metaclass=StrictBossMeta):
    # The developer writes it in lowercase
    salary = 50000  


# 3. WHAT HAPPENS WHEN WE RUN IT?
bob = Employee()

# Mypy will say this next line is an ERROR, but Python will run it perfectly!
print(bob.SALARY)  # Output: 50000

# The original lowercase 'salary' literally doesn't exist anymore!
# print(bob.salary) -> AttributeError

50000
